# PBMC 10k: Rust vs Python Inference Speed

Benchmarks `fit_all_grid_points` on `processed_pbmc_10k_raw.h5ad` (10,997 cells × 36,601 genes)
comparing the Rust PSS fast-path against the pure-Python baseline.

**Model:** Bursty + Poisson  
Three sweeps:
1. **Core-count sweep** — fixed 50 genes, vary `num_cores` 1→8
2. **Gene-count sweep** — fixed `num_cores=4`, vary genes 25→200
3. **Cell-count sweep** — fixed `num_cores=4`, 50 genes, subsample cells 100→10k

Inference uses a reduced 3×4 sampling grid (12 points) and 10 optimizer iterations
to keep runtimes practical while preserving relative differences.

## Setup

In [1]:
# Work from the repository root (data in example_h5ad/, figures in figures/)
# and import monod from src/ when running against a checkout.
import os, sys
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, 'setup.cfg')) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root)
if os.path.isdir(os.path.join(_root, 'src')) and os.path.join(_root, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(_root, 'src'))


In [2]:
import matplotlib
matplotlib.use("Agg")
import gc
import sys, os, time, warnings, tempfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import anndata as ad

warnings.filterwarnings('ignore')

import monod.cme_toolbox as cme_toolbox
import monod.inference as inference
from monod.cme_toolbox import CMEModel, _HAS_RUST
from monod.extract_data import extract_data
from monod.inference import InferenceParameters, searchdata_from_adata
os.makedirs('figures', exist_ok=True)

print(f'Rust extension available: {_HAS_RUST}')
print(f'Logical CPUs: {os.cpu_count()}')

Rust extension available: True
Logical CPUs: 24


## Constants and helpers

In [3]:
H5AD_PATH = 'example_h5ad/processed_pbmc_10k_raw.h5ad'
MODEL     = CMEModel('Bursty', 'Poisson')

# Reduced grid and iteration count to keep runtimes practical.
GRADIENT_PARAMS_BASE = {
    'max_iterations': 10,
    'init_pattern': 'moments',
    'num_restarts': 1,
}
GRIDSIZE = [3, 4]  # 12 sampling points instead of the default 6×7=42

def time_full_pipeline(h5ad_path, n_genes, num_cores, has_rust, rng_seed=0):
    """Time the full pipeline: extract_data + fit_all_grid_points."""
    gp = dict(GRADIENT_PARAMS_BASE, num_gene_cores=num_cores)
    cme_toolbox._HAS_RUST = has_rust
    inference._HAS_RUST   = has_rust
    np.random.seed(rng_seed)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        t0 = time.perf_counter()
        adata = extract_data(
            h5ad_path, MODEL,
            dataset_name='pbmc_bench',
            modality_name_dict={'unspliced': 'unspliced', 'spliced': 'spliced'},
            n_genes=n_genes, hist_type='unique', viz=False,
        )
        sd = searchdata_from_adata(adata)
        ip = InferenceParameters('pbmc_bench', MODEL,
                                 use_lengths=False, gradient_params=gp,
                                 gridsize=GRIDSIZE, save=False)
        ip.fit_all_grid_points(sd, num_cores=num_cores, save=False)
        elapsed = time.perf_counter() - t0
    cme_toolbox._HAS_RUST = _HAS_RUST
    inference._HAS_RUST   = _HAS_RUST
    return elapsed


## 1. Core-count sweep

Fixed 50 genes; vary `num_cores`.

In [4]:
CORE_COUNTS  = [1, 2, 4, 8]
N_GENES_CORE = 50

core_results = {}

for nc in CORE_COUNTS:
    for has_rust in (True, False):
        print(f'cores={nc}, rust={has_rust} ...', end=' ', flush=True)
        t = time_full_pipeline(H5AD_PATH, N_GENES_CORE, nc, has_rust)
        core_results[(nc, has_rust)] = t
        print(f'{t:.1f}s')


cores=1, rust=True ... 

Grid scan:   0%|          | 0/12 [00:00<?, ?it/s]

Grid scan:   8%|▊         | 1/12 [00:00<00:06,  1.73it/s]

Grid scan:  17%|█▋        | 2/12 [00:01<00:05,  1.73it/s]

Grid scan:  25%|██▌       | 3/12 [00:01<00:05,  1.71it/s]

...

Grid scan:  83%|████████▎ | 10/12 [00:55<00:08,  4.22s/it]

Grid scan:  92%|█████████▏| 11/12 [00:58<00:03,  3.84s/it]

Grid scan: 100%|██████████| 12/12 [01:02<00:00,  3.91s/it]

Grid scan: 100%|██████████| 12/12 [01:02<00:00,  5.23s/it]

67.3s


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

rust_t   = [core_results[(nc, True)]  for nc in CORE_COUNTS]
python_t = [core_results[(nc, False)] for nc in CORE_COUNTS]
speedups = [core_results[(nc, False)] / core_results[(nc, True)] for nc in CORE_COUNTS]

x, w = np.arange(len(CORE_COUNTS)), 0.35
ax = axes[0]
ax.bar(x - w/2, rust_t,   w, label='Rust',   color='steelblue')
ax.bar(x + w/2, python_t, w, label='Python', color='coral')
ax.set_xticks(x); ax.set_xticklabels(CORE_COUNTS)
ax.set_xlabel('num_cores'); ax.set_ylabel('Wall time (s)')
ax.set_title(f'Core-count sweep ({N_GENES_CORE} genes)')
ax.legend()

ax = axes[1]
ax.plot(CORE_COUNTS, speedups, 'o-', color='steelblue', linewidth=2)
ax.axhline(1, linestyle='--', color='gray')
ax.set_xlabel('num_cores'); ax.set_ylabel('Rust speedup')
ax.set_title('Rust speedup vs num_cores')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
fig.tight_layout()
plt.savefig(f"figures/pbmc_core_sweep.png", dpi=72, bbox_inches="tight")
plt.close("all")
gc.collect()

2427

## 2. Gene-count sweep

Fixed `num_cores=4`; vary number of genes.

In [6]:
GENE_COUNTS  = [25, 50, 100, 200]
SWEEP_CORES  = 4
gene_results = {}

for n_genes in GENE_COUNTS:
    for has_rust in (True, False):
        print(f'n_genes={n_genes}, rust={has_rust} ...', end=' ', flush=True)
        t = time_full_pipeline(H5AD_PATH, n_genes, SWEEP_CORES, has_rust)
        gene_results[(n_genes, has_rust)] = t
        print(f'{t:.1f}s')


n_genes=25, rust=True ... 

Grid scan:   0%|          | 0/12 [00:00<?, ?it/s]

Grid scan:   8%|▊         | 1/12 [00:00<00:01,  6.27it/s]

Grid scan:  17%|█▋        | 2/12 [00:00<00:01,  6.71it/s]

Grid scan:  25%|██▌       | 3/12 [00:00<00:01,  7.06it/s]

...

Grid scan:  83%|████████▎ | 10/12 [03:56<00:36, 18.48s/it]

Grid scan:  92%|█████████▏| 11/12 [04:10<00:17, 17.20s/it]

Grid scan: 100%|██████████| 12/12 [04:29<00:00, 17.68s/it]

Grid scan: 100%|██████████| 12/12 [04:29<00:00, 22.45s/it]

273.9s


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

gr_rust    = [gene_results[(n, True)]  for n in GENE_COUNTS]
gr_python  = [gene_results[(n, False)] for n in GENE_COUNTS]
gr_speedup = [gene_results[(n, False)] / gene_results[(n, True)] for n in GENE_COUNTS]

ax = axes[0]
ax.plot(GENE_COUNTS, gr_rust,   'o-', color='steelblue', label='Rust')
ax.plot(GENE_COUNTS, gr_python, 's-', color='coral',     label='Python')
ax.set_xlabel('Number of genes'); ax.set_ylabel('Wall time (s)')
ax.set_title(f'Gene-count sweep (num_cores={SWEEP_CORES})')
ax.legend()

ax = axes[1]
ax.plot(GENE_COUNTS, gr_speedup, 'o-', color='steelblue', linewidth=2)
ax.axhline(1, linestyle='--', color='gray')
ax.set_xlabel('Number of genes'); ax.set_ylabel('Rust speedup')
ax.set_title('Rust speedup vs gene count')
fig.tight_layout()
plt.savefig(f"figures/pbmc_gene_sweep.png", dpi=72, bbox_inches="tight")
plt.close("all")
gc.collect()

6301

## 3. Cell-count sweep

Fixed `num_cores=4`, 50 genes; subsample cells to vary histogram density.
Top x-axis shows median PSS grid size at each cell count.

In [8]:
CELL_COUNTS  = [100, 500, 1000, 10000]
SWEEP_GENES  = 50
cell_results = {}  # (n_cells, has_rust) -> (elapsed, median_grid)

# Pre-create all subsampled temp files before timing,
# then free the full adata to avoid OOM during inference.
full_adata = ad.read_h5ad(H5AD_PATH)
rng = np.random.default_rng(0)
tmp_paths = {}
for n_cells in CELL_COUNTS:
    idx = rng.choice(full_adata.n_obs, size=min(n_cells, full_adata.n_obs), replace=False)
    sub = full_adata[idx].copy()
    with tempfile.NamedTemporaryFile(suffix='.h5ad', delete=False) as f:
        tmp_paths[n_cells] = f.name
    sub.write_h5ad(tmp_paths[n_cells])
    del sub
    gc.collect()
del full_adata
gc.collect()

for n_cells in CELL_COUNTS:
    tmp = tmp_paths[n_cells]
    # Probe M values (outside the timed region).
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        adata_probe = extract_data(tmp, MODEL,
            dataset_name='pbmc_probe',
            modality_name_dict={'unspliced': 'unspliced', 'spliced': 'spliced'},
            n_genes=SWEEP_GENES, hist_type='unique', viz=False)
    median_grid = int(np.median(adata_probe.uns['M'][0] * adata_probe.uns['M'][1]))
    del adata_probe
    for has_rust in (True, False):
        print(f'n_cells={n_cells}, rust={has_rust} (median grid={median_grid}) ...', end=' ', flush=True)
        t = time_full_pipeline(tmp, SWEEP_GENES, SWEEP_CORES, has_rust)
        cell_results[(n_cells, has_rust)] = (t, median_grid)
        print(f'{t:.1f}s')
    os.unlink(tmp)


n_cells=100, rust=True (median grid=289) ... 

Grid scan:   0%|          | 0/12 [00:00<?, ?it/s]

Grid scan:   8%|▊         | 1/12 [00:00<00:02,  3.90it/s]

Grid scan:  17%|█▋        | 2/12 [00:00<00:02,  4.59it/s]

Grid scan:  25%|██▌       | 3/12 [00:00<00:01,  4.64it/s]

...

Grid scan:  83%|████████▎ | 10/12 [00:51<00:08,  4.05s/it]

Grid scan:  92%|█████████▏| 11/12 [00:55<00:03,  3.78s/it]

Grid scan: 100%|██████████| 12/12 [00:58<00:00,  3.63s/it]

Grid scan: 100%|██████████| 12/12 [00:58<00:00,  4.86s/it]

62.4s


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

cr_rust    = [cell_results[(n, True)][0]  for n in CELL_COUNTS]
cr_python  = [cell_results[(n, False)][0] for n in CELL_COUNTS]
cr_speedup = [cell_results[(n, False)][0] / cell_results[(n, True)][0] for n in CELL_COUNTS]
cr_grids   = [cell_results[(n, True)][1]  for n in CELL_COUNTS]

ax = axes[0]
ax.plot(CELL_COUNTS, cr_rust,   'o-', color='steelblue', label='Rust')
ax.plot(CELL_COUNTS, cr_python, 's-', color='coral',     label='Python')
ax.set_xscale('log')
ax.set_xlabel('Number of cells'); ax.set_ylabel('Wall time (s)')
ax.set_title(f'Cell-count sweep (num_cores={SWEEP_CORES}, {SWEEP_GENES} genes)')
ax.legend()

ax = axes[1]
ax.set_xscale('log')
ax.plot(CELL_COUNTS, cr_speedup, 'o-', color='steelblue', linewidth=2)
ax.axhline(1, linestyle='--', color='gray')
ax2 = ax.twiny()
ax2.set_xscale('log')
ax2.set_xlim(ax.get_xlim())
ax2.set_xticks(CELL_COUNTS)
ax2.set_xticklabels([f'M~{g}' for g in cr_grids], fontsize=8)
ax.set_xlabel('Number of cells'); ax.set_ylabel('Rust speedup')
ax.set_title('Rust speedup vs cell count')
fig.tight_layout()
plt.savefig(f"figures/pbmc_cell_sweep.png", dpi=72, bbox_inches="tight")
plt.close("all")
gc.collect()

15171